In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc
import pandas as pd

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\output\example_trajectories")
save_path.mkdir(parents=True, exist_ok=True)

include = [1, 4, 6, 7, 8, 10, 11, 12, 13, 14]
condition = [0, 0, 0, 0, 0, 1, 2, 2, 2, 2]
earliest_frames = [25, 43, 63, 80, 35, 23, 36, 17, 5, 3]

dnt.set_plot_style()

spots_dfs, metadatas, stems = dnt.load_spots_data(spots_directory, include)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

In [ ]:
region_colors = dnt.palettes.ap[0:5:2]
regions = ["Anterior", "Middle", "Posterior"]

for k, df in enumerate(spots_dfs):

    df = df.copy()
    df = df[df["frame"] >= earliest_frames[k]]
    spots_dfs[k] = df

    t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "cycle"]].mean().reset_index()

    t["AP_from_start"] = t["AP"] - t["track_id"].map(t.groupby("track_id")["AP"].first())
    t["time_since_nc11"] = np.round(t["time_since_nc11"], 3)

    if condition[k] == 1:
        t["time_since_nc11"] = t["time_since_nc11"] / 60
        nc_11_time = df[df["cycle"] == 11].groupby("track_id")["time_since_nc11"].min().mean()
        t["time_since_nc11"] = t["time_since_nc11"] - nc_11_time

    fig, axes = plt.subplots(figsize=(7, 4))

    for i, ap_group in enumerate([(0.05, 0.2), (0.4, 0.6), (0.8, 0.95)]):

        region_track_ids = t[t["AP"].between(*ap_group)]["track_id"].unique()
        early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
        early_track_ids = t["track_id"].unique()[early_track_ids]

        print("Region", region_track_ids)
        print(early_track_ids)
        print(t["cycle"].unique())
        # low_distance = t.groupby("track_id")["distance"].mean() < 3.0
        # low_distance = t["track_id"].unique()[low_distance]

        all_good = np.intersect1d(region_track_ids, early_track_ids)

        np.random.seed(42)
        sampled_tracks = np.random.choice(all_good, 50)

        t_new = t[t["track_id"].isin(sampled_tracks)].copy()

        sns.lineplot(t_new, x="time_since_nc11", y="AP_from_start", color=region_colors[i], errorbar=None, alpha=1, label=regions[i], linewidth=5)

        for cycle in cycles[1:]:
            cycle_df = t[t["cycle"] == cycle]
            cycle_times = cycle_df.groupby("track_id")["time_since_nc11"].min()
            division_time = cycle_times.median()
            plt.axvline(division_time, color="k", linestyle="--", linewidth=2, alpha=0.2)

    plt.title(f"{stems[k]} example trajectories in posterior region")
    plt.ylabel("AP position")
    plt.xlabel("Time since nc11 (minutes)")
    plt.legend()
    # plt.ylim(-0.03, 0.13)
    plt.savefig(save_path / f"{stems[k]}_example_trajectories_three_regions.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
k = 0

for k, df in enumerate(spots_dfs):
    df = df.copy()
    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)

    df = df[df["frame"] > min_mvmt_frames[0]]

    frame_map = lambda f: (f - df["frame"].min()) * metadatas[k]["seconds_per_frame"] / 60.0
    df["time"] = (df["frame"] - df["frame"].min()) * metadatas[k]["seconds_per_frame"] / 60.0

    t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "distance", "cycle_pseudotime", "cycle", "time"]].mean().reset_index()
    t["time"] = np.round(t["time"], 3)
    t["AP_from_start"] = t["AP"] - t["track_id"].map(t.groupby("track_id")["AP"].first())

    # identify track ids that start early and aren't too weird
    early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
    early_track_ids = t["track_id"].unique()[early_track_ids]
    low_distance = t.groupby("track_id")["distance"].mean() < 3.0
    low_distance = t["track_id"].unique()[low_distance]

    for region, bounds in zip(["anterior", "middle", "posterior"], [(0.1, 0.2), (0.45, 0.55), (0.8, 0.9)]):

        region_track_ids = t[t["AP"].between(*bounds)]["track_id"].unique()
        all_good = np.intersect1d(np.intersect1d(region_track_ids, early_track_ids), low_distance)

        np.random.seed(42)
        t_region = t[t["track_id"].isin(all_good)].copy()

        division_times = []

        # indentify division times within the region
        for i, nc in enumerate(cycles):

            t_nc = t_region[t_region["frame"] == min_mvmt_frames[i]].set_index("track_id")

            if nc < 14:
                cycle_df = t_region[t_region["cycle"] == nc]
                cycle_df = cycle_df[cycle_df["distance"] < 3.0]
                cycle_times = cycle_df.groupby("track_id")["frame"].min()

            else:
                cycle_df = t_region[t_region["cycle"] == 13]
                cycle_df = cycle_df[cycle_df["distance"] < 3.0]
                cycle_times = cycle_df.groupby("track_id")["frame"].max()

            division_times.append(cycle_times.mean())

        fig, ax = plt.subplots(figsize=(5, 4))

        for frame in division_times:
            plt.axvline(frame_map(frame), color="k", linestyle="--", linewidth=2)

        sampled_tracks = np.random.choice(all_good, 10)
        t_ss = t[t["track_id"].isin(sampled_tracks)].copy()

        sns.lineplot(t_ss, x="time", y="AP_from_start", hue="track_id", errorbar=None, palette="crest", alpha=0.2, lw=2)
        sns.lineplot(t_region, x="time", y="AP_from_start", errorbar=None, color = sns.color_palette("crest", as_cmap=True)(0.5), lw=3)

        plt.gca().get_legend().remove()

        plt.title(f"{region.capitalize()} region")
        plt.ylabel("Total AP movement")
        plt.xlabel("Time (minutes)")
        plt.ylim(-0.05, 0.10)

        plt.savefig(save_path / f"{stems[k]}_example_trajectories_{region}_region.png", dpi=300, bbox_inches="tight")

        plt.close()

In [ ]:
import napari
df = spots_dfs[-2]
viewer = napari.Viewer(ndisplay=3)
color = [dnt.palettes.nc[cycle] for cycle in df["cycle"].values]
viewer.add_points(df[["frame", "z", "y", "x"]].values, size=df["radius"]*2.2, face_color=color, border_color="k", border_width=0.1)
napari.run()

In [ ]:
k = 5
df = spots_dfs[k]

min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "distance", "cycle_pseudotime", "cycle"]].mean().reset_index()

# returns the next mvmt frame for a given frame
map_next = lambda x: min_mvmt_frames[np.searchsorted(min_mvmt_frames, x, side="right")] if x < max(min_mvmt_frames) else np.nan
map_prev = lambda x: min_mvmt_frames[np.searchsorted(min_mvmt_frames, x, side="left") -1] if x > min(min_mvmt_frames) else np.nan

t["next_stop"] = t["frame"].map(map_next)
t["prev_stop"] = t["frame"].map(map_prev)

t["frames_to_next"] = t["next_stop"] - t["frame"]
t["frames_from_prev"] = t["frame"] - t["prev_stop"]

t["dist_to_next"] = 0.
t["dist_from_prev"] = 0.
t["gap_distance"] = 0.

region_track_ids = t[t["AP"] > 0.8]["track_id"].unique()
early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
early_track_ids = t["track_id"].unique()[early_track_ids]
low_distance = t.groupby("track_id")["distance"].mean() < 3.0
low_distance = t["track_id"].unique()[low_distance]

all_good = np.intersect1d(np.intersect1d(region_track_ids, early_track_ids), low_distance)

np.random.seed(42)
sampled_tracks = np.random.choice(all_good, 10)


t = t[t["track_id"].isin(sampled_tracks)].copy()

division_times = []

print(t[["frame", "next_stop", "prev_stop"]])

for i, nc in enumerate(cycles):

    t_nc = t[t["frame"] == min_mvmt_frames[i]].set_index("track_id")

    for axis in ["x", "y", "z", "AP"]:
        t[f"d{nc}{axis}"] = t[axis] - t["track_id"].map(t_nc[axis])

    t[f"{nc}_dist"] = np.sqrt(t[f"d{nc}x"]**2 + t[f"d{nc}y"]**2 + t[f"d{nc}z"]**2)

    # sns.lineplot(t, x="frame", y=f"{nc}_dist", errorbar=None, color=dnt.palettes.nc[nc], label=nc)

    if nc < 14:
        cycle_df = t[t["cycle"] == nc]
        cycle_df = cycle_df[cycle_df["distance"] < 3.0]
        cycle_times = cycle_df.groupby("track_id")["frame"].min()

    else:
        cycle_df = t[t["cycle"] == 13]
        cycle_df = cycle_df[cycle_df["distance"] < 3.0]
        cycle_times = cycle_df.groupby("track_id")["frame"].max()

    division_times.append(cycle_times.mean())

    if i < len(cycles) - 1:
        t_nc_next = t[t["frame"] == min_mvmt_frames[i + 1]].set_index("track_id")
        t[f"{nc}-{nc+1}"] = t["track_id"].map(t_nc_next[f"{nc}_dist"])

    if i > 0:
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "dist_from_prev"] = t[f"{nc - 1}_dist"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "dist_to_next"] = t[f"{nc}_dist"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "gap_distance"] = t[f"{nc-1}-{nc}"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "AP_from_prev"] = t[f"d{nc -1}AP"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "AP_to_next"] = t[f"d{nc}AP"]

    else:
        t.loc[t["frame"] <= min_mvmt_frames[i], "dist_to_next"] = t[f"{nc}_dist"]


print(division_times)

(save_path / stems[k]).mkdir(exist_ok=True)

t["sum_distance"] = t["dist_to_next"] + t["dist_from_prev"]
t["track_id"] = pd.Categorical(t["track_id"])

sns.lineplot(t, x="frame", y="dist_from_prev", hue="track_id", errorbar=None, palette="crest")
for frame in min_mvmt_frames:
    plt.axvline(frame, color="k", linestyle="-", linewidth=2)
plt.legend(title="track id")
plt.ylabel("Distance from previous stop (um)")
plt.xlabel("frame")
plt.savefig(save_path  / stems[k] / "nc_movement_distance_from_previous_stop.png", dpi=300)
plt.show()

sns.lineplot(t, x="frame", y="dist_to_next", hue="track_id", errorbar=None, palette="flare")
for frame in min_mvmt_frames:
    plt.axvline(frame, color="k", linestyle="-", linewidth=2)
plt.legend(title="track id")
plt.ylabel("Distance to next stop (um)")
plt.xlabel("frame")
plt.savefig(save_path  / stems[k] / "nc_movement_distance_to_next_stop.png", dpi=300)
plt.show()

sns.lineplot(t, x="frame", y="AP_from_prev", hue="track_id", errorbar=None, palette="crest")
for frame in min_mvmt_frames:
    plt.axvline(frame, color="k", linestyle="-", linewidth=2)
for frame in division_times:
    plt.axvline(frame, color="r", linestyle="-", linewidth=1)
plt.gca().get_legend().remove()
plt.ylabel("AP from prev stop (fraction")
plt.xlabel("frame")
plt.savefig(save_path / stems[k] / "nc_movement_ap_from_prev.png", dpi=300)
plt.show()


## Bad and quick

In [ ]:
k = 4
df = spots_dfs[k]

min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "distance", "cycle_pseudotime", "cycle"]].mean().reset_index()

# returns the next mvmt frame for a given frame
map_next = lambda x: min_mvmt_frames[np.searchsorted(min_mvmt_frames, x, side="right")] if x < max(min_mvmt_frames) else np.nan
map_prev = lambda x: min_mvmt_frames[np.searchsorted(min_mvmt_frames, x, side="left") -1] if x > min(min_mvmt_frames) else np.nan

t["next_stop"] = t["frame"].map(map_next)
t["prev_stop"] = t["frame"].map(map_prev)

t["frames_to_next"] = t["next_stop"] - t["frame"]
t["frames_from_prev"] = t["frame"] - t["prev_stop"]

t["dist_to_next"] = 0.
t["dist_from_prev"] = 0.
t["gap_distance"] = 0.

region_track_ids = t[t["AP"].between(0.05, 0.95)]["track_id"].unique()
early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
early_track_ids = t["track_id"].unique()[early_track_ids]
low_distance = t.groupby("track_id")["distance"].mean() < 3.0
low_distance = t["track_id"].unique()[low_distance]

all_good = np.intersect1d(np.intersect1d(region_track_ids, early_track_ids), low_distance)

np.random.seed(42)
sampled_tracks = np.random.choice(all_good, 50)


t = t[t["track_id"].isin(sampled_tracks)].copy()

division_times = []

print(t[["frame", "next_stop", "prev_stop"]])

for i, nc in enumerate(cycles):

    t_nc = t[t["frame"] == min_mvmt_frames[i]].set_index("track_id")

    for axis in ["x", "y", "z", "AP"]:
        t[f"d{nc}{axis}"] = t[axis] - t["track_id"].map(t_nc[axis])

    t[f"{nc}_dist"] = np.sqrt(t[f"d{nc}x"]**2 + t[f"d{nc}y"]**2 + t[f"d{nc}z"]**2)

    if nc < 14:
        cycle_df = t[t["cycle"] == nc]
        cycle_df = cycle_df[cycle_df["distance"] < 3.0]
        cycle_times = cycle_df.groupby("track_id")["frame"].min()

    else:
        cycle_df = t[t["cycle"] == 13]
        cycle_df = cycle_df[cycle_df["distance"] < 3.0]
        cycle_times = cycle_df.groupby("track_id")["frame"].max()

    division_times.append(cycle_times.mean())

    if i < len(cycles) - 1:
        t_nc_next = t[t["frame"] == min_mvmt_frames[i + 1]].set_index("track_id")
        t[f"{nc}-{nc+1}"] = t["track_id"].map(t_nc_next[f"{nc}_dist"])

    if i > 0:
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "dist_from_prev"] = t[f"{nc - 1}_dist"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "dist_to_next"] = t[f"{nc}_dist"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "gap_distance"] = t[f"{nc-1}-{nc}"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "AP_from_prev"] = t[f"d{nc -1}AP"]
        t.loc[t["frame"].between(min_mvmt_frames[i - 1], min_mvmt_frames[i]), "AP_to_next"] = t[f"d{nc}AP"]

    else:
        t.loc[t["frame"] <= min_mvmt_frames[i], "dist_to_next"] = t[f"{nc}_dist"]

(save_path / stems[k]).mkdir(exist_ok=True)

t["track_id"] = pd.Categorical(t["track_id"])

sns.lineplot(t, x="frame", y="AP_from_prev", errorbar=None, color = sns.color_palette("crest", as_cmap=True)(0.5))
for frame in min_mvmt_frames:
    plt.axvline(frame, color="k", linestyle="-", linewidth=2)
for frame in division_times:
    plt.axvline(frame, color="r", linestyle="-", linewidth=1)

plt.ylabel("AP from prev stop (fraction")
plt.xlabel("frame")
plt.savefig(save_path / stems[k] / "all_nc_movement_ap_from_prev.png", dpi=300)
plt.show()
